In [1]:
from pathlib import Path

PROJECT_ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data").is_dir() and (path / "notebooks").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the project directory.")

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "hdb_resale_clean.csv")
df["month"] = pd.to_datetime(df["month"])

In [2]:
df["log_price"] = np.log(df["resale_price"])

In [3]:
model = smf.ols(
    formula="""
        log_price ~
        floor_area_sqm
        + remaining_lease_years_numeric
        + storey_midpoint
        + C(town)
        + C(flat_type)
        + C(year)
    """,
    data=df
).fit(cov_type="HC3")

In [4]:
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              log_price   R-squared:                       0.907
Model:                            OLS   Adj. R-squared:                  0.907
Method:                 Least Squares   F-statistic:                 5.603e+04
Date:                Wed, 09 Sep 2026   Prob (F-statistic):               0.00
Time:                        20:19:11   Log-Likelihood:             1.9524e+05
No. Observations:              239330   AIC:                        -3.904e+05
Df Residuals:                  239286   BIC:                        -3.899e+05
Df Model:                          43                                         
Covariance Type:                  HC3                                         
                                       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercep

/Users/sney/Projects/singapore-hdb-market-analytics/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: divide by zero encountered in matmul
  return _core_matmul(x1, x2)
/Users/sney/Projects/singapore-hdb-market-analytics/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: overflow encountered in matmul
  return _core_matmul(x1, x2)
/Users/sney/Projects/singapore-hdb-market-analytics/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: invalid value encountered in matmul
  return _core_matmul(x1, x2)


In [5]:
core_vars = [
    "floor_area_sqm",
    "remaining_lease_years_numeric",
    "storey_midpoint"
]

results = pd.DataFrame({
    "coefficient": model.params[core_vars],
    "std_error": model.bse[core_vars],
    "p_value": model.pvalues[core_vars]
})

results["pct_effect"] = (
    (np.exp(results["coefficient"]) - 1) * 100
)

results

,coefficient,std_error,p_value,pct_effect
floor_area_sqm,0.008328,0.000054,0.0,0.836321
remaining_lease_years_numeric,0.010278,0.000021,0.0,1.033129
storey_midpoint,0.007894,0.000042,0.0,0.792569


### Core Continuous Effects

Holding town, flat type, year, and the other included characteristics constant:

- Each additional square metre of floor area is associated with approximately 0.84% higher resale price.
- Each additional year of remaining lease is associated with approximately 1.03% higher resale price.
- Each additional storey level is associated with approximately 0.79% higher resale price.

All three associations are statistically significant at conventional levels.

In [6]:
town_effects = model.params[
    model.params.index.str.startswith("C(town)")
]

town_effects_pct = (
    (np.exp(town_effects) - 1) * 100
).sort_values(ascending=False)

town_effects_pct.head(10)

C(town)[T.BUKIT TIMAH]        34.979888
C(town)[T.CENTRAL AREA]       29.692617
C(town)[T.MARINE PARADE]      28.160908
C(town)[T.BISHAN]             17.304135
C(town)[T.QUEENSTOWN]         16.864982
C(town)[T.BUKIT MERAH]        15.681313
C(town)[T.KALLANG/WHAMPOA]     9.269697
C(town)[T.TOA PAYOH]           6.204640
C(town)[T.CLEMENTI]            5.491363
C(town)[T.GEYLANG]             4.235942
dtype: float64

In [7]:
sorted(df["town"].unique())

['ANG MO KIO',
 'BEDOK',
 'BISHAN',
 'BUKIT BATOK',
 'BUKIT MERAH',
 'BUKIT PANJANG',
 'BUKIT TIMAH',
 'CENTRAL AREA',
 'CHOA CHU KANG',
 'CLEMENTI',
 'GEYLANG',
 'HOUGANG',
 'JURONG EAST',
 'JURONG WEST',
 'KALLANG/WHAMPOA',
 'MARINE PARADE',
 'PASIR RIS',
 'PUNGGOL',
 'QUEENSTOWN',
 'SEMBAWANG',
 'SENGKANG',
 'SERANGOON',
 'TAMPINES',
 'TOA PAYOH',
 'WOODLANDS',
 'YISHUN']

In [8]:
flat_type_effects = model.params[
    model.params.index.str.startswith("C(flat_type)")
]

flat_type_effects_pct = (
    (np.exp(flat_type_effects) - 1) * 100
).sort_values(ascending=False)

flat_type_effects_pct

C(flat_type)[T.MULTI-GENERATION]    61.554218
C(flat_type)[T.EXECUTIVE]           50.344115
C(flat_type)[T.5 ROOM]              42.607116
C(flat_type)[T.4 ROOM]              40.891729
C(flat_type)[T.3 ROOM]              31.715445
C(flat_type)[T.2 ROOM]              12.965296
dtype: float64

In [9]:
print("R-squared:", round(model.rsquared, 3))
print("Adjusted R-squared:", round(model.rsquared_adj, 3))

R-squared: 0.907
Adjusted R-squared: 0.907


In [10]:
df["log_price_per_sqm"] = np.log(df["price_per_sqm"])

In [11]:
model_psm = smf.ols(
    formula="""
        log_price_per_sqm ~
        remaining_lease_years_numeric
        + storey_midpoint
        + C(town)
        + C(flat_type)
        + C(year)
    """,
    data=df
).fit(cov_type="HC3")

In [12]:
psm_core_vars = [
    "remaining_lease_years_numeric",
    "storey_midpoint"
]

psm_results = pd.DataFrame({
    "coefficient": model_psm.params[psm_core_vars],
    "p_value": model_psm.pvalues[psm_core_vars]
})

psm_results["pct_effect"] = (
    (np.exp(psm_results["coefficient"]) - 1) * 100
)

psm_results

,coefficient,p_value,pct_effect
remaining_lease_years_numeric,0.010263,0.0,1.031614
storey_midpoint,0.007959,0.0,0.799109


In [13]:
town_effects = model.params[
    model.params.index.str.startswith("C(town)")
]

town_effects_pct = (
    (np.exp(town_effects) - 1) * 100
).sort_values(ascending=False)

town_effects_pct.head(10)

C(town)[T.BUKIT TIMAH]        34.979888
C(town)[T.CENTRAL AREA]       29.692617
C(town)[T.MARINE PARADE]      28.160908
C(town)[T.BISHAN]             17.304135
C(town)[T.QUEENSTOWN]         16.864982
C(town)[T.BUKIT MERAH]        15.681313
C(town)[T.KALLANG/WHAMPOA]     9.269697
C(town)[T.TOA PAYOH]           6.204640
C(town)[T.CLEMENTI]            5.491363
C(town)[T.GEYLANG]             4.235942
dtype: float64

In [14]:
sorted(df["town"].unique())

['ANG MO KIO',
 'BEDOK',
 'BISHAN',
 'BUKIT BATOK',
 'BUKIT MERAH',
 'BUKIT PANJANG',
 'BUKIT TIMAH',
 'CENTRAL AREA',
 'CHOA CHU KANG',
 'CLEMENTI',
 'GEYLANG',
 'HOUGANG',
 'JURONG EAST',
 'JURONG WEST',
 'KALLANG/WHAMPOA',
 'MARINE PARADE',
 'PASIR RIS',
 'PUNGGOL',
 'QUEENSTOWN',
 'SEMBAWANG',
 'SENGKANG',
 'SERANGOON',
 'TAMPINES',
 'TOA PAYOH',
 'WOODLANDS',
 'YISHUN']

In [15]:
flat_type_effects = model.params[
    model.params.index.str.startswith("C(flat_type)")
]

flat_type_effects_pct = (
    (np.exp(flat_type_effects) - 1) * 100
).sort_values(ascending=False)

flat_type_effects_pct

C(flat_type)[T.MULTI-GENERATION]    61.554218
C(flat_type)[T.EXECUTIVE]           50.344115
C(flat_type)[T.5 ROOM]              42.607116
C(flat_type)[T.4 ROOM]              40.891729
C(flat_type)[T.3 ROOM]              31.715445
C(flat_type)[T.2 ROOM]              12.965296
dtype: float64

In [16]:
sorted(df["flat_type"].unique())

['1 ROOM',
 '2 ROOM',
 '3 ROOM',
 '4 ROOM',
 '5 ROOM',
 'EXECUTIVE',
 'MULTI-GENERATION']

### Robustness Check

Using log price per square metre as the dependent variable produced very similar results.

- Each additional year of remaining lease was associated with approximately 1.03% higher price per sqm.
- Each additional storey level was associated with approximately 0.80% higher price per sqm.

The consistency of these estimates with the main resale-price model suggests that the observed relationships are not driven solely by differences in flat size.